# DP single-phase inverter state-space extraction

This notebook validates native-DP state-space extraction for the single-phase DP averaged voltage-source inverter and compares it with the same EMT inverter case in the transformed `GlobalDQ0` analysis frame.

Topology:

```text
GND -- Inverter -- nInv -- R -- nMid -- L -- nGrid -- VoltageSource -- GND
```

The DP and EMT simulations use the same topology, parameters, time step, operating point, and gain scales as `EMT_Ph3_Inverter_StateSpaceExtraction.ipynb`.

The notebook performs two checks:

1. **Native DP extraction check**
   - Compares DPsim-extracted native-DP eigenvalues with an analytical dq-frame model.
   - The analytical reference is the 14-state dq model, without zero-sequence states.
   - The assertion is applied to discrete eigenvalues `z`, because the `z -> lambda` mapping can amplify small differences for fast modes.

2. **Modal and time-domain comparison**
   - For the stable and unstable gain scales, plots:
     - discrete DP DPsim eigenvalues and discrete EMT DPsim eigenvalues in the transformed `GlobalDQ0` frame,
     - continuous DP DPsim eigenvalues, continuous EMT DPsim eigenvalues in the transformed `GlobalDQ0` frame, and analytical dq-frame eigenvalues,
     - DP and EMT active-power responses.

The EMT `GlobalDQ0` spectrum contains additional zero-sequence electrical modes that are not represented by the DP Ph1 model or by the analytical dq-frame reference. These extra EMT modes are expected and are not treated as a DP/analytical mismatch.


In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
dp = dpsimpy.dp

ph3 = emt.ph3
dp_ph1 = dp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
Solver = dpsimpy.Solver
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis
StateSpaceAnalysisFrame = dpsimpy.StateSpaceAnalysisFrame

## Parameters

The values below intentionally match the EMT three-phase inverter state-space extraction notebook.


In [38]:
# Simulation settings
time_step = 10e-6
final_time_short = time_step
final_time_validation = 0.6

# System
system_frequency = 50.0
omega0 = 2.0 * np.pi * system_frequency

# Fixed ideal source voltage
grid_voltage_rms_ll = 400.0
source_phase_deg = -90.0

# Grid branch
r_grid = 0.3
l_grid = 1.0e-3

# Inverter filter
lf = 2.0e-3
cf = 10.0e-6
rf = 0.2
rc = 0.2

# PLL
kp_pll = 0.25
ki_pll = 0.2
omega_cutoff = omega0

# Power references
p_ref = 10000.0
q_ref = 5000.0

# Base power controller
kp_power_ctrl_base = 0.05
ki_power_ctrl_base = 0.2

# Base current controller
kp_curr_ctrl_base = 0.25
ki_curr_ctrl_base = 1.0

# Gain scales close to the stability boundary
gain_scale_stable = 3.7
gain_scale_unstable = 3.8

sim_name_base = "DP_Ph1_Inverter_StateSpaceExtraction"

## Common utilities


In [39]:
abc_scale = np.sqrt(2.0 / 3.0)


def abc_phasor(phase_a_phasor):
    return np.array(
        [
            [phase_a_phasor],
            [phase_a_phasor * np.exp(-1j * 2.0 * np.pi / 3.0)],
            [phase_a_phasor * np.exp(+1j * 2.0 * np.pi / 3.0)],
        ],
        dtype=np.complex128,
    )


def abc_instantaneous_from_phasor(phase_a_phasor, t=0.0):
    return np.real(
        abc_scale * abc_phasor(phase_a_phasor).reshape(3) * np.exp(1j * omega0 * t)
    )


def three_phase_param(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def source_phasor():
    return grid_voltage_rms_ll * np.exp(1j * np.deg2rad(source_phase_deg))


def park_matrix(theta):
    k = np.sqrt(2.0 / 3.0)

    return np.array(
        [
            [
                k * np.cos(theta),
                k * np.cos(theta - 2.0 * np.pi / 3.0),
                k * np.cos(theta + 2.0 * np.pi / 3.0),
            ],
            [
                -k * np.sin(theta),
                -k * np.sin(theta - 2.0 * np.pi / 3.0),
                -k * np.sin(theta + 2.0 * np.pi / 3.0),
            ],
        ],
        dtype=float,
    )


def numerical_jacobian(f, x0, eps=1e-6):
    x0 = np.asarray(x0, dtype=float)
    f0 = np.asarray(f(x0), dtype=float)
    A = np.zeros((len(f0), len(x0)))

    for k in range(len(x0)):
        h = eps * max(1.0, abs(x0[k]))

        xp = x0.copy()
        xm = x0.copy()

        xp[k] += h
        xm[k] -= h

        A[:, k] = (f(xp) - f(xm)) / (2.0 * h)

    return A


def trapezoidal_discretization(A, dt):
    I = np.eye(A.shape[0])

    return np.linalg.solve(
        I - 0.5 * dt * A,
        I + 0.5 * dt * A,
    )


def bilinear_lambda_from_z(z, dt):
    return (2.0 / dt) * ((z - 1.0) / (z + 1.0))


def finite_complex(values):
    values = np.asarray(values).reshape(-1)

    return values[np.isfinite(values.real) & np.isfinite(values.imag)]


def closest_eigenvalue_matches(reference, candidate):
    rows = []

    for value_ref in reference:
        idx = np.argmin(np.abs(candidate - value_ref))
        rows.append(
            {
                "reference": value_ref,
                "closest": candidate[idx],
                "distance": abs(candidate[idx] - value_ref),
            }
        )

    return sorted(rows, key=lambda row: row["distance"], reverse=True)


def symmetric_max_eigenvalue_distance(a, b):
    a = finite_complex(a)
    b = finite_complex(b)

    return max(
        closest_eigenvalue_matches(a, b)[0]["distance"],
        closest_eigenvalue_matches(b, a)[0]["distance"],
    )


def max_eigenvalue_distance_to_reference(values, reference):
    values = finite_complex(values)
    reference = finite_complex(reference)

    return max(np.min(np.abs(reference - value)) for value in values)


def is_discrete_stable(values):
    return bool(np.max(np.abs(finite_complex(values))) < 1.0)


def is_continuous_stable(values):
    return bool(np.max(finite_complex(values).real) < 0.0)


def eigenvalue_table(lambdas, source, n=None):
    values = finite_complex(lambdas)
    order = np.argsort(values.real)[::-1]

    if n is not None:
        order = order[:n]

    values = values[order]

    return pd.DataFrame(
        {
            "source": source,
            "Re(lambda) [1/s]": values.real,
            "Im(lambda) [rad/s]": values.imag,
            "frequency [Hz]": np.abs(values.imag) / (2.0 * np.pi),
        }
    )


def complex_to_state(value):
    return np.array([value.real, value.imag], dtype=float)


def state_to_complex(x, offset):
    return x[offset] + 1j * x[offset + 1]


def dq_to_complex(value_dq, theta):
    return value_dq * np.exp(1j * theta)


def complex_to_dq(value, theta):
    return value * np.exp(-1j * theta)

## Operating point

The operating point is the same one used in the EMT state-space extraction notebook. The inverter power references are interpreted at the inverter capacitor voltage.


In [40]:
def steady_state_operating_point(gain_scale=1.0):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    S = p_ref + 1j * q_ref
    E = source_phasor()

    Z_total = rc + r_grid + 1j * omega0 * l_grid
    C = Z_total * np.conj(S)

    roots = np.roots(
        [
            1.0,
            -(abs(E) ** 2 + 2.0 * C.real),
            abs(C) ** 2,
        ]
    )

    roots_real = np.real(roots[np.isreal(roots)])
    y = np.max(roots_real[roots_real > 0.0])

    Vc = (y - np.conj(C)) / np.conj(E)
    Iinj = np.conj(S / Vc)

    Uinv = Vc - rc * Iinj
    Umid = E + 1j * omega0 * l_grid * Iinj

    If = Iinj + 1j * omega0 * cf * Vc
    Vref = Vc + (rf + 1j * omega0 * lf) * If

    theta = np.angle(Vc)

    vc_abc = abc_instantaneous_from_phasor(Vc)
    if_abc = abc_instantaneous_from_phasor(If)
    i_line_abc = abc_instantaneous_from_phasor(Iinj)
    u_inv_abc = abc_instantaneous_from_phasor(Uinv)
    u_mid_abc = abc_instantaneous_from_phasor(Umid)
    e_grid_abc = abc_instantaneous_from_phasor(E)
    v_ref_abc = abc_instantaneous_from_phasor(Vref)

    T = park_matrix(theta)

    vc_dq = T @ vc_abc
    i_dq = T @ i_line_abc
    v_ref_dq = T @ v_ref_abc

    vc_d, vc_q = vc_dq
    i_d, i_q = i_dq

    p0 = vc_d * i_d + vc_q * i_q
    q0 = -vc_d * i_q + vc_q * i_d

    phi_pll = 0.0
    p_filtered = p0
    q_filtered = q0

    phi_d = (i_d - kp_power_ctrl * (p_ref - p_filtered)) / ki_power_ctrl
    phi_q = (i_q - kp_power_ctrl * (q_filtered - q_ref)) / ki_power_ctrl

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    gamma_d = (v_ref_dq[0] - kp_curr_ctrl * (i_ref_d - i_d)) / ki_curr_ctrl
    gamma_q = (v_ref_dq[1] - kp_curr_ctrl * (i_ref_q - i_q)) / ki_curr_ctrl

    x_emt_inv = np.zeros(14)
    x_emt_inv[0] = theta
    x_emt_inv[1] = phi_pll
    x_emt_inv[2] = p_filtered
    x_emt_inv[3] = q_filtered
    x_emt_inv[4] = phi_d
    x_emt_inv[5] = phi_q
    x_emt_inv[6] = gamma_d
    x_emt_inv[7] = gamma_q
    x_emt_inv[8:11] = vc_abc
    x_emt_inv[11:14] = if_abc

    x_emt_full = np.zeros(17)
    x_emt_full[:14] = x_emt_inv
    x_emt_full[14:17] = i_line_abc

    x_dp_full = np.zeros(14)
    x_dp_full[0] = theta
    x_dp_full[1] = phi_pll
    x_dp_full[2] = p_filtered
    x_dp_full[3] = q_filtered
    x_dp_full[4] = phi_d
    x_dp_full[5] = phi_q
    x_dp_full[6] = gamma_d
    x_dp_full[7] = gamma_q
    x_dp_full[8:10] = complex_to_state(Vc)
    x_dp_full[10:12] = complex_to_state(If)
    x_dp_full[12:14] = complex_to_state(Iinj)

    return {
        "E": E,
        "Vc": Vc,
        "Uinv": Uinv,
        "Umid": Umid,
        "Iinj": Iinj,
        "If": If,
        "Vref": Vref,
        "theta": theta,
        "x_emt_inv": x_emt_inv,
        "x_emt_full": x_emt_full,
        "x_dp_full": x_dp_full,
        "vc_abc": vc_abc,
        "if_abc": if_abc,
        "i_line_abc": i_line_abc,
        "u_inv_abc": u_inv_abc,
        "u_mid_abc": u_mid_abc,
        "e_grid_abc": e_grid_abc,
    }

## Analytical dq-frame model

The analytical model uses the same inverter, controller, filter, and grid equations as the DP simulation, written in dq coordinates around the fixed operating point.

It contains the 14 positive-sequence/dq states that correspond to the DP Ph1 native model:

```text
theta, phi_pll, p_filtered, q_filtered, phi_d, phi_q, gamma_d, gamma_q,
vc_d, vc_q, if_d, if_q, i_line_d, i_line_q
```

No zero-sequence block is added to this analytical reference. Therefore, the EMT `GlobalDQ0` result can contain additional zero-sequence modes that are not expected to appear in either the DP Ph1 or analytical dq spectra.


In [41]:
def inverter_rhs_native_dp(x_inv, u_inv, gain_scale):
    kp_power_ctrl = gain_scale * kp_power_ctrl_base
    ki_power_ctrl = gain_scale * ki_power_ctrl_base
    kp_curr_ctrl = gain_scale * kp_curr_ctrl_base
    ki_curr_ctrl = gain_scale * ki_curr_ctrl_base

    theta = x_inv[0]
    phi_pll = x_inv[1]
    p_filtered = x_inv[2]
    q_filtered = x_inv[3]
    phi_d = x_inv[4]
    phi_q = x_inv[5]
    gamma_d = x_inv[6]
    gamma_q = x_inv[7]

    Vc = state_to_complex(x_inv, 8)
    If = state_to_complex(x_inv, 10)

    Irc = (Vc - u_inv) / rc

    vc_dq = complex_to_dq(Vc, theta)
    i_dq = complex_to_dq(Irc, theta)

    vc_d = vc_dq.real
    vc_q = vc_dq.imag
    i_d = i_dq.real
    i_q = i_dq.imag

    p_inst = vc_d * i_d + vc_q * i_q
    q_inst = -vc_d * i_q + vc_q * i_d

    omega_pll = omega0 + kp_pll * vc_q + ki_pll * phi_pll

    i_ref_d = kp_power_ctrl * (p_ref - p_filtered) + ki_power_ctrl * phi_d
    i_ref_q = kp_power_ctrl * (q_filtered - q_ref) + ki_power_ctrl * phi_q

    v_ref_d = kp_curr_ctrl * (i_ref_d - i_d) + ki_curr_ctrl * gamma_d
    v_ref_q = kp_curr_ctrl * (i_ref_q - i_q) + ki_curr_ctrl * gamma_q

    Vref = dq_to_complex(v_ref_d + 1j * v_ref_q, theta)

    dVc = If / cf + (u_inv - Vc) / (cf * rc) - 1j * omega0 * Vc
    dIf = (Vref - Vc - rf * If) / lf - 1j * omega0 * If

    dx = np.zeros(12)
    dx[0] = omega_pll - omega0
    dx[1] = vc_q
    dx[2] = omega_cutoff * (p_inst - p_filtered)
    dx[3] = omega_cutoff * (q_inst - q_filtered)
    dx[4] = p_ref - p_filtered
    dx[5] = q_filtered - q_ref
    dx[6] = i_ref_d - i_d
    dx[7] = i_ref_q - i_q
    dx[8:10] = complex_to_state(dVc)
    dx[10:12] = complex_to_state(dIf)

    return dx


def full_rhs_native_dp(x_full, op, gain_scale):
    x_inv = x_full[:12]
    Iline = state_to_complex(x_full, 12)

    Vc = state_to_complex(x_inv, 8)
    Uinv = Vc - rc * Iline

    dx_inv = inverter_rhs_native_dp(x_inv, Uinv, gain_scale)
    dIline = (Uinv - r_grid * Iline - op["E"]) / l_grid - 1j * omega0 * Iline

    dx = np.zeros(14)
    dx[:12] = dx_inv
    dx[12:14] = complex_to_state(dIline)

    return dx


def complex_envelope_to_dq_jacobian(value, theta):
    value_dq = value * np.exp(-1j * theta)

    d = value_dq.real
    q = value_dq.imag

    c = np.cos(theta)
    s = np.sin(theta)

    J_block = np.array(
        [
            [c, s],
            [-s, c],
        ],
        dtype=float,
    )

    theta_column = np.array([q, -d], dtype=float)

    return J_block, theta_column


def native_dp_to_dq_jacobian(op):
    J = np.eye(14)
    theta = op["theta"]

    for value, offset in [
        (op["Vc"], 8),
        (op["If"], 10),
        (op["Iinj"], 12),
    ]:
        J_block, theta_column = complex_envelope_to_dq_jacobian(value, theta)

        J[offset : offset + 2, offset : offset + 2] = J_block
        J[offset : offset + 2, 0] = theta_column

    return J


def analytical_dq_frame_model(op, gain_scale):
    A_native = numerical_jacobian(
        lambda x: full_rhs_native_dp(x, op, gain_scale),
        op["x_dp_full"],
    )

    J = native_dp_to_dq_jacobian(op)
    J_inv = np.linalg.inv(J)

    A_dq = J @ A_native @ J_inv
    Ad_dq = trapezoidal_discretization(A_dq, time_step)

    return {
        "A": A_dq,
        "Ad": Ad_dq,
        "z": np.linalg.eigvals(Ad_dq),
        "lambda": np.linalg.eigvals(A_dq),
    }

## DPsim modal and time-domain setup


In [42]:
def get_native_modal_results(sim):
    extractor = sim.get_state_space_extractor()

    modal = StateSpaceModalAnalysis(extractor)
    modal.update()

    return {
        "Ad": np.array(extractor.get_discrete_state_matrix(), copy=True),
        "z": np.array(modal.get_discrete_eigenvalues(), copy=True).reshape(-1),
        "lambda": np.array(modal.get_continuous_eigenvalues(), copy=True).reshape(-1),
        "state_count": extractor.get_state_count(),
    }


def get_global_dq0_modal_results(sim, theta0=0.0):
    extractor = sim.get_state_space_extractor()

    modal = StateSpaceModalAnalysis(extractor)
    modal.set_analysis_frame(StateSpaceAnalysisFrame.GlobalDQ0)
    modal.set_global_dq0_frame(omega0, theta0)
    modal.update()

    return {
        "Ad": np.array(extractor.get_discrete_state_matrix(), copy=True),
        "z": np.array(modal.get_discrete_eigenvalues(), copy=True).reshape(-1),
        "lambda": np.array(modal.get_continuous_eigenvalues(), copy=True).reshape(-1),
        "state_count": extractor.get_state_count(),
    }


def load_log(sim_name):
    return pd.read_csv(
        Path("logs") / sim_name / f"{sim_name}.csv",
        skipinitialspace=True,
    )


def signal(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]

    return df[cols].to_numpy()


def scalar_signal(df, name):
    return signal(df, name)[:, 0]


def time_vector(df):
    return df.iloc[:, 0].to_numpy()


def build_emt_dpsim_system(op, gain_scale):
    n_inv = emt.SimNode("nInv", PhaseType.ABC)
    n_mid = emt.SimNode("nMid", PhaseType.ABC)
    n_grid = emt.SimNode("nGrid", PhaseType.ABC)

    n_inv.set_initial_voltage(op["Uinv"])
    n_mid.set_initial_voltage(op["Umid"])
    n_grid.set_initial_voltage(op["E"])

    voltage_source = ph3.VoltageSource("VoltageSource")
    voltage_source.set_parameters(
        abc_phasor(op["E"]),
        system_frequency,
    )

    resistor = ph3.Resistor("GridResistor")
    resistor.set_parameters(three_phase_param(r_grid))

    inductor = ph3.Inductor("GridInductor")
    inductor.set_parameters(three_phase_param(l_grid))

    inverter = ph3.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        omega0,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        gain_scale * kp_power_ctrl_base,
        gain_scale * ki_power_ctrl_base,
        gain_scale * kp_curr_ctrl_base,
        gain_scale * ki_curr_ctrl_base,
    )

    inverter.connect([emt.SimNode.gnd, n_inv])
    resistor.connect([n_inv, n_mid])
    inductor.connect([n_mid, n_grid])
    voltage_source.connect([emt.SimNode.gnd, n_grid])

    system = SystemTopology(
        system_frequency,
        [n_inv, n_mid, n_grid],
        [inverter, resistor, inductor, voltage_source],
    )

    return system, inverter


def build_dp_dpsim_system(op, gain_scale):
    n_inv = dp.SimNode("nInv", PhaseType.Single)
    n_mid = dp.SimNode("nMid", PhaseType.Single)
    n_grid = dp.SimNode("nGrid", PhaseType.Single)

    n_inv.set_initial_voltage(op["Uinv"])
    n_mid.set_initial_voltage(op["Umid"])
    n_grid.set_initial_voltage(op["E"])

    voltage_source = dp_ph1.VoltageSource("VoltageSource")

    # Constant native-DP envelope. Do not rotate the envelope again.
    voltage_source.set_parameters(op["E"], 0.0)

    resistor = dp_ph1.Resistor("GridResistor")
    resistor.set_parameters(r_grid)

    inductor = dp_ph1.Inductor("GridInductor")
    inductor.set_parameters(l_grid)

    inverter = dp_ph1.AvVoltSourceInverterStateSpace("INV_SSN_PLL")
    inverter.set_parameters(
        lf,
        cf,
        rf,
        rc,
        omega0,
        kp_pll,
        ki_pll,
        omega_cutoff,
        p_ref,
        q_ref,
        gain_scale * kp_power_ctrl_base,
        gain_scale * ki_power_ctrl_base,
        gain_scale * kp_curr_ctrl_base,
        gain_scale * ki_curr_ctrl_base,
    )

    inverter.connect([dp.SimNode.gnd, n_inv])
    resistor.connect([n_inv, n_mid])
    inductor.connect([n_mid, n_grid])
    voltage_source.connect([dp.SimNode.gnd, n_grid])

    system = SystemTopology(
        system_frequency,
        [n_inv, n_mid, n_grid],
        [inverter, resistor, inductor, voltage_source],
    )

    return system, inverter

In [43]:
def run_emt_modal_case(gain_scale, case_name, skip_periods=1):
    period = 1.0 / system_frequency
    steps_per_period = int(round(period / time_step))
    total_steps = (skip_periods + 1) * steps_per_period

    sim_name = f"{sim_name_base}_EMT_{case_name}_modal_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_emt_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(total_steps * time_step)

    global_dq0_modal = None

    sim.start()

    for step_idx in range(total_steps):
        sim.next()

        if step_idx >= skip_periods * steps_per_period and global_dq0_modal is None:
            global_dq0_modal = get_global_dq0_modal_results(sim, theta0=0.0)

    sim.stop()

    if global_dq0_modal is None:
        raise RuntimeError("GlobalDQ0 modal results were not collected.")

    z = finite_complex(global_dq0_modal["z"])
    lambdas = finite_complex(global_dq0_modal["lambda"])

    return {
        "op": op,
        "modal": global_dq0_modal,
        "z": z,
        "lambda": lambdas,
        "state_count": global_dq0_modal["state_count"],
        "max_abs_z": np.max(np.abs(z)),
        "max_re_lambda": np.max(lambdas.real),
        "stable": is_discrete_stable(z),
    }


def run_dp_modal_case(gain_scale, case_name):
    sim_name = f"{sim_name_base}_DP_{case_name}_modal_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_dp_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.do_state_space_extraction(True)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time_short)

    modal_first = None

    sim.start()

    sim.next()
    modal_first = get_native_modal_results(sim)

    sim.stop()

    analytical = analytical_dq_frame_model(op, gain_scale)

    z = finite_complex(modal_first["z"])
    lambdas = finite_complex(modal_first["lambda"])

    return {
        "op": op,
        "modal": modal_first,
        "analytical": analytical,
        "z": z,
        "lambda": lambdas,
        "state_count": modal_first["state_count"],
        "max_abs_z": np.max(np.abs(z)),
        "max_re_lambda": np.max(lambdas.real),
        "stable": is_discrete_stable(z),
        "z_error_to_analytical_dq": symmetric_max_eigenvalue_distance(
            z,
            analytical["z"],
        ),
        "lambda_error_to_analytical_dq": max_eigenvalue_distance_to_reference(
            lambdas,
            analytical["lambda"],
        ),
    }


def run_emt_time_domain(gain_scale, case_name):
    sim_name = f"{sim_name_base}_EMT_{case_name}_time_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_emt_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    logger = Logger(sim_name)
    logger.log_attribute("p_inst", inverter.attr("p_inst"))
    logger.log_attribute("q_inst", inverter.attr("q_inst"))
    logger.log_attribute("vc_d", inverter.attr("vc_d"))
    logger.log_attribute("vc_q", inverter.attr("vc_q"))
    logger.log_attribute("omega_pll", inverter.attr("omega_pll"))
    sim.add_logger(logger)

    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time_validation)
    sim.run()

    return {
        "sim_name": sim_name,
        "op": op,
    }


def run_dp_time_domain(gain_scale, case_name):
    sim_name = f"{sim_name_base}_DP_{case_name}_time_gain_{gain_scale:g}".replace(
        ".", "_"
    )
    Logger.set_log_dir("logs/" + sim_name)

    op = steady_state_operating_point(gain_scale)
    system, inverter = build_dp_dpsim_system(op, gain_scale)

    sim = Simulation(sim_name)
    sim.set_system(system)

    logger = Logger(sim_name)
    logger.log_attribute("p_inst", inverter.attr("p_inst"))
    logger.log_attribute("q_inst", inverter.attr("q_inst"))
    logger.log_attribute("vc_d", inverter.attr("vc_d"))
    logger.log_attribute("vc_q", inverter.attr("vc_q"))
    logger.log_attribute("omega_pll", inverter.attr("omega_pll"))
    sim.add_logger(logger)

    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)

    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)

    sim.set_time_step(time_step)
    sim.set_final_time(final_time_validation)
    sim.run()

    return {
        "sim_name": sim_name,
        "op": op,
    }

## Check 1: native DP frame against analytical dq-frame model

The DP extracted matrix is in the native DP frame. Since the operating point is fixed in this frame, its eigenvalues can be compared directly with the analytical dq-frame eigenvalues.

The assertion is applied to the 14-state DP spectrum and the 14-state analytical dq-frame spectrum. The EMT `GlobalDQ0` result is shown for comparison, but it is not part of this assertion because it also contains zero-sequence electrical modes.


In [ ]:
modal_stable = {
    "case": "stable",
    "gain_scale": gain_scale_stable,
    "expected_stable": True,
    "emt": run_emt_modal_case(gain_scale_stable, "stable"),
    "dp": run_dp_modal_case(gain_scale_stable, "stable"),
}

modal_unstable = {
    "case": "unstable",
    "gain_scale": gain_scale_unstable,
    "expected_stable": False,
    "emt": run_emt_modal_case(gain_scale_unstable, "unstable"),
    "dp": run_dp_modal_case(gain_scale_unstable, "unstable"),
}

modal_cases = [modal_stable, modal_unstable]

dp_z_tolerance = 2e-3

for result in modal_cases:
    assert result["dp"]["state_count"] == 14
    assert len(result["dp"]["analytical"]["z"]) == 14
    assert result["dp"]["z_error_to_analytical_dq"] < dp_z_tolerance

    assert result["dp"]["stable"] == result["expected_stable"]
    assert result["emt"]["stable"] == result["expected_stable"]
    assert (
        is_continuous_stable(result["dp"]["analytical"]["lambda"])
        == result["expected_stable"]
    )

print("Check 1 passed.")

pd.DataFrame(
    [
        {
            "case": result["case"],
            "gain scale": result["gain_scale"],
            "DP state count": result["dp"]["state_count"],
            "analytical dq state count": len(result["dp"]["analytical"]["z"]),
            "EMT GlobalDQ0 state count": result["emt"]["state_count"],
            "extra EMT modes": result["emt"]["state_count"]
            - len(result["dp"]["analytical"]["z"]),
            "DP stable": result["dp"]["stable"],
            "EMT GlobalDQ0 stable": result["emt"]["stable"],
            "Analytical dq stable": is_continuous_stable(
                result["dp"]["analytical"]["lambda"]
            ),
            "max |z|, DP native": result["dp"]["max_abs_z"],
            "max |z|, EMT GlobalDQ0": result["emt"]["max_abs_z"],
            "max Re(lambda), DP native [1/s]": result["dp"]["max_re_lambda"],
            "max Re(lambda), EMT GlobalDQ0 [1/s]": result["emt"]["max_re_lambda"],
            "max |Δz|, DP vs analytical dq": result["dp"]["z_error_to_analytical_dq"],
            "max nearest |Δlambda|, DP vs analytical dq [1/s]": result["dp"][
                "lambda_error_to_analytical_dq"
            ],
        }
        for result in modal_cases
    ]
)

In [ ]:
pd.concat(
    [
        eigenvalue_table(
            modal_stable["dp"]["lambda"],
            "DPsim DP native, stable",
            n=8,
        ),
        eigenvalue_table(
            modal_stable["dp"]["analytical"]["lambda"],
            "Analytical dq, stable",
            n=8,
        ),
    ],
    ignore_index=True,
)

## Check 2: plots for stable and unstable cases

For each gain scale, the plots show:

1. discrete eigenvalues from DPsim DP native extraction and DPsim EMT `GlobalDQ0` extraction,
2. continuous eigenvalues from DPsim DP native extraction, DPsim EMT `GlobalDQ0` extraction, and the 14-state analytical dq-frame model,
3. active-power responses from DPsim DP and DPsim EMT simulations.

The same color/marker convention is used in all eigenvalue plots:

- DPsim DP native: open circles,
- DPsim EMT `GlobalDQ0`: open squares,
- analytical dq: crosses.

The EMT `GlobalDQ0` plots contain three additional zero-sequence modes. These are expected because the EMT `dq0` representation includes `vc_0`, `if_0`, and `i_line_0`, while the DP Ph1 and analytical dq models contain only the positive-sequence/dq dynamics.


In [ ]:
time_stable = {
    "case": "stable",
    "gain_scale": gain_scale_stable,
    "emt": run_emt_time_domain(gain_scale_stable, "stable"),
    "dp": run_dp_time_domain(gain_scale_stable, "stable"),
}

time_unstable = {
    "case": "unstable",
    "gain_scale": gain_scale_unstable,
    "emt": run_emt_time_domain(gain_scale_unstable, "unstable"),
    "dp": run_dp_time_domain(gain_scale_unstable, "unstable"),
}

df_emt_stable = load_log(time_stable["emt"]["sim_name"])
df_dp_stable = load_log(time_stable["dp"]["sim_name"])

df_emt_unstable = load_log(time_unstable["emt"]["sim_name"])
df_dp_unstable = load_log(time_unstable["dp"]["sim_name"])

In [47]:
DP_STYLE = {
    "marker": "o",
    "facecolors": "none",
    "edgecolors": "C0",
    "s": 60,
    "linewidths": 1.3,
}

EMT_STYLE = {
    "marker": "s",
    "facecolors": "none",
    "edgecolors": "C1",
    "s": 90,
    "linewidths": 1.3,
}

ANALYTICAL_STYLE = {
    "marker": "x",
    "color": "black",
    "s": 30,
    "linewidths": 1.3,
}


def plot_discrete_eigenvalues(ax, dp_z, emt_z, title):
    dp_z = finite_complex(dp_z)
    emt_z = finite_complex(emt_z)

    phi = np.linspace(0.0, 2.0 * np.pi, 500)

    ax.plot(
        np.cos(phi),
        np.sin(phi),
        linestyle="--",
        linewidth=0.8,
        color="0.5",
        label="unit circle",
    )

    ax.scatter(
        dp_z.real,
        dp_z.imag,
        **DP_STYLE,
        label="DPsim DP native z",
    )

    ax.scatter(
        emt_z.real,
        emt_z.imag,
        **EMT_STYLE,
        label="DPsim EMT GlobalDQ0 z",
    )

    ax.axhline(0.0, linewidth=0.8, color="0.5")
    ax.axvline(0.0, linewidth=0.8, color="0.5")

    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True)
    ax.legend()


def plot_continuous_eigenvalues(ax, dp_lambda, emt_lambda, analytical_lambda, title):
    dp_lambda = finite_complex(dp_lambda)
    emt_lambda = finite_complex(emt_lambda)
    analytical_lambda = finite_complex(analytical_lambda)

    ax.scatter(
        dp_lambda.real,
        dp_lambda.imag,
        **DP_STYLE,
        label="DPsim DP native",
    )

    ax.scatter(
        emt_lambda.real,
        emt_lambda.imag,
        **EMT_STYLE,
        label="DPsim EMT GlobalDQ0",
    )

    ax.scatter(
        analytical_lambda.real,
        analytical_lambda.imag,
        **ANALYTICAL_STYLE,
        label="analytical dq",
    )

    ax.axhline(0.0, linewidth=0.8, color="0.5")
    ax.axvline(0.0, linewidth=0.8, color="0.5")

    ax.set_xlabel("Re(λ) [1/s]")
    ax.set_ylabel("Im(λ) [rad/s]")
    ax.set_title(title)
    ax.grid(True)
    ax.legend()


def plot_power_response(ax, df_dp, df_emt, title):
    t_dp = time_vector(df_dp)
    p_dp = scalar_signal(df_dp, "p_inst")

    t_emt = time_vector(df_emt)
    p_emt = scalar_signal(df_emt, "p_inst")

    ax.set_ylim(p_ref * 0.99, p_ref * 1.01)

    ax.plot(t_dp, p_dp, color="C0", linestyle="--", label="DPsim DP p_inst")
    ax.plot(t_emt, p_emt, color="C1", label="DPsim EMT p_inst")

    ax.axhline(
        p_ref,
        color="C2",
        linestyle=":",
        linewidth=1.0,
        label="p_ref",
    )

    ax.set_xlabel("time [s]")
    ax.set_ylabel("active power [W]")
    ax.set_title(title)
    ax.ticklabel_format(axis="y", style="plain", useOffset=False)
    ax.grid(True)
    ax.legend()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, modal_result, time_result, df_dp, df_emt in [
    (0, modal_stable, time_stable, df_dp_stable, df_emt_stable),
    (1, modal_unstable, time_unstable, df_dp_unstable, df_emt_unstable),
]:
    case = modal_result["case"]
    gain_scale = modal_result["gain_scale"]

    plot_discrete_eigenvalues(
        axes[row, 0],
        modal_result["dp"]["z"],
        modal_result["emt"]["z"],
        f"Discrete eigenvalues, {case}, gain = {gain_scale:g}",
    )

    plot_continuous_eigenvalues(
        axes[row, 1],
        modal_result["dp"]["lambda"],
        modal_result["emt"]["lambda"],
        modal_result["dp"]["analytical"]["lambda"],
        f"Continuous eigenvalues, {case}, gain = {gain_scale:g}",
    )

    plot_power_response(
        axes[row, 2],
        df_dp,
        df_emt,
        f"Active power, {case}, gain = {gain_scale:g}",
    )

plt.tight_layout()
plt.show()